In [1]:
from abc import ABC, abstractmethod

In [2]:
class Runable(ABC):

  @abstractmethod
  def invoke(input_data):
    pass

In [3]:
import random
class DupLLM:

  def __init__ (self):
    print('LLM created')

  def invoke(self, prompt):
    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]

    return {'response': random.choice(response_list)}

    def predict(self, prompt):
      response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
      ]

      return {'response': random.choice(response_list)}


In [5]:
class DupPromptTemplate(Runable):

  def __init__(self, template, input_variables):
    self.template = template
    self.input_variables = input_variables

  def invoke(self, input_dict):
    return self.template.format(**input_dict)

  def format(self, input_dict):
    return self.template.format(**input_dict)

In [6]:
class DupOutputParser(Runable):

  def __init__(self):
    pass

  def invoke(self, input_data):
    return input_data['response']


In [7]:
class RunableConnector(Runable):
  def __init__(self, runnable_list):
    self.runnable_list = runnable_list

  def invoke(self, input_data):
    for runnable in self.runnable_list:
      input_data = runnable.invoke(input_data)
    return input_data

In [28]:
template = DupPromptTemplate(
    template='Write a {length} poem about the {topic}',
    input_variables=['length', 'topic']
)

In [9]:
output_parser = DupOutputParser()

In [10]:
llm = DupLLM()

LLM created


In [11]:
connector = RunableConnector(
    runnable_list=[template, llm, output_parser]
)

In [12]:
connector.invoke({'length': 'long', 'topic': 'Cricket'})

'AI stands for Artificial Intelligence'

In [14]:
template1 = DupPromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

In [15]:
template2 = DupPromptTemplate(
    template='Explain the following joke {response}',
    input_variables=['response']
)

In [16]:
llm = DupLLM()

LLM created


In [17]:
parser = DupOutputParser()

In [18]:
chain1 = RunableConnector(
    runnable_list=[template1, llm, parser]
)


In [25]:
chain2 = RunableConnector(
    runnable_list=[StringWrapper(key_name='response'), template2, llm, parser]
)

In [26]:
final_chain = RunableConnector(
    runnable_list=[chain1, chain2]
)

In [27]:
final_chain.invoke({'length':'long', 'topic': 'Cricket'})

'Delhi is the capital of India'

In [23]:
class StringWrapper(Runable):
  def __init__(self, key_name):
    self.key_name = key_name
  def invoke(self, input_data):
    if isinstance(input_data, str):
      return {self.key_name: input_data}
    return input_data # Pass through if it's already a dict